In [0]:
storage_account = "insurancestorage01"

spark.conf.set(
"fs.azure.account.key."+storage_account+".dfs.core.windows.net",
"azure_Storage_key"
)

spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

silver_adls_path = "abfss://silver@"+storage_account+".dfs.core.windows.net/"
gold_adls_path = "abfss://gold@"+storage_account+".dfs.core.windows.net/"

In [0]:
from pyspark.sql.functions import col

hospitals = spark.read.format("delta").load(silver_adls_path+"/hospitals")

dim_hospital = (
    hospitals
    .select(
        "hospital_id",
        "hospital_name",
        "hospital_type",
        "network_flag",
        "city",
        "state",
        "region",
        "rating"
    )
    .dropDuplicates(["hospital_id"])
)

dim_hospital.write.format("delta").mode("overwrite").save(gold_adls_path+"/dim_hospitals")
